<a href="https://colab.research.google.com/github/slomi23/NLP_final_project/blob/main/notebooks/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import importlib
from pathlib import Path

project_root = Path("/content/NLP_final_project").resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("Project exists:", project_root.exists())

if not project_root.exists():
    raise FileNotFoundError("Repo not found. Run the clone cell first.")

import src.models.encoder as encoder_module
import src.models.loss as loss_module

importlib.reload(encoder_module)
importlib.reload(loss_module)

EncoderOnlyTransformer = encoder_module.EncoderOnlyTransformer
SimpleTokenizer = encoder_module.SimpleTokenizer
InfoNCELoss = loss_module.InfoNCELoss

print("✅ Reloaded fixed encoder/loss")
print("Files in project root:")
for p in project_root.iterdir():
    print(" -", p.name)


Project root: C:\content\NLP_final_project
Project exists: False


FileNotFoundError: Repo not found. Run the clone cell first.

In [ ]:
%cd /content/NLP_final_project
!pip -q install pypdf

PDF_PATH = project_root / "data" / "raw" / "Speech_and_Language_Processing.pdf"
print("PDF exists:", PDF_PATH.exists(), PDF_PATH)

if not PDF_PATH.exists():
    raise FileNotFoundError(f"Missing Jurafsky PDF: {PDF_PATH}")

!rm -f data/jurafsky_chunks/chunks.jsonl
!python src/data/book_chunks.py


📚 Loading ArXiv papers from c:\Users\salo\NLP_final_project\data\processed\arxiv_cs_papers_processed.csv...
✅ Loaded 417331 ArXiv papers
✅ Prepared 417331 ArXiv samples

📊 Total Training Dataset:
   - Jurafsky Chunks: 1730
   - ArXiv Papers:    417331
   - Total:           419061
📏 Average word count per sample: 188.7


In [ ]:
import json
import numpy as np

CHUNKS_FILE = project_root / "data" / "jurafsky_chunks" / "chunks.jsonl"

if not CHUNKS_FILE.exists():
    raise FileNotFoundError(CHUNKS_FILE)

chunks = []
with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line))

print(f"✅ Loaded {len(chunks)} clean Jurafsky chunks")
word_counts = [len(c["text"].split()) for c in chunks]
print(f"Word count min/mean/max: {min(word_counts)} / {np.mean(word_counts):.1f} / {max(word_counts)}")

for i, c in enumerate(chunks[:3]):
    print("\n" + "=" * 90)
    print("CHUNK", i, "word_count:", len(c["text"].split()))
    print(c["text"][:1200])


🔄 Running make_pairs.py to generate training pairs...
Loading data вЂ¦
Building pairs вЂ¦
  total pairs: 20,000
  train/val/test: 16000/2000/2000
  wrote 16,000 pairs в†’ data\processed\train_pairs.jsonl
  wrote 2,000 pairs в†’ data\processed\val_pairs.jsonl
  wrote 2,000 pairs в†’ data\processed\test_pairs.jsonl
Done вњ“

📚 Loading training pairs from c:\Users\salo\NLP_final_project\data\processed\train_pairs.jsonl...
✅ Loaded 16000 training pairs
✅ Prepared 16000 samples from pairs

📊 Final Combined Training Dataset:
   - Jurafsky Chunks: 1730
   - ArXiv Papers:    417331
   - MS MARCO Pairs:  16000
   - TOTAL:           435061
✅ Data ready for training!


In [ ]:
import json
import random
import re
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load MS MARCO Data Directly
print("📚 Loading MS MARCO data directly from JSON files...")

QUERIES_FILE = project_root / "data" / "processed" / "msmarco_small_queries.json"
PASSAGES_FILE = project_root / "data" / "processed" / "msmarco_small_passages.json"
QRELS_FILE = project_root / "data" / "processed" / "msmarco_small_qrels.json"

with open(QUERIES_FILE, 'r', encoding='utf-8') as f:
    queries_list = json.load(f)

with open(PASSAGES_FILE, 'r', encoding='utf-8') as f:
    passages_list = json.load(f)

with open(QRELS_FILE, 'r', encoding='utf-8') as f:
    qrels = json.load(f)

print(f"✅ Loaded {len(queries_list)} queries")
print(f"✅ Loaded {len(passages_list)} passages")
print(f"✅ Loaded {len(qrels)} relevance judgments")

# 2. Create Lookups
query_lookup = {q['id']: q['text'] for q in queries_list}
passage_lookup = {p['id']: p['text'] for p in passages_list}

# 3. Generate MS MARCO Triplets
print("🔄 Generating MS MARCO triplets...")
ms_marco_triplets = []

for query_id, relevant_passages in qrels.items():
    if query_id not in query_lookup:
        continue

    query_text = query_lookup[query_id]

    # Get the best relevant passage (highest score)
    best_pid, _ = max(relevant_passages, key=lambda x: x)

    if best_pid not in passage_lookup:
        continue

    positive_text = passage_lookup[best_pid]

    # Filter short queries
    if len(query_text.split()) < 3:
        continue

    ms_marco_triplets.append({
        'query': query_text,
        'positive': positive_text,
        'negatives': [] # Will add negatives later
    })

print(f"✅ Generated {len(ms_marco_triplets)} MS MARCO pairs")

# 4. Load ArXiv Triplets
print("📚 Loading ArXiv triplets...")
ARXIV_TRAIN_FILE = project_root / "data" / "processed" / "arxiv_train_triplets.jsonl"
arxiv_triplets = []
with open(ARXIV_TRAIN_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            arxiv_triplets.append(json.loads(line.strip()))

print(f"✅ Loaded {len(arxiv_triplets)} ArXiv triplets")

# 5. Generate Jurafsky Book Triplets (Synthetic)
print("📚 Generating Jurafsky Book triplets...")

# Load Jurafsky chunks
chunks_file = project_root / "data" / "jurafsky_chunks" / "chunks.jsonl"
chunks = []
with open(chunks_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line.strip()))

# Helper to extract sentences
def extract_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    valid_sentences = [
        s.strip() for s in sentences
        if len(s.split()) > 5 and not s.startswith('(') and not s.startswith('[')
    ]
    return valid_sentences

jurafsky_triplets = []
for chunk in chunks:
    text = chunk['text']
    sentences = extract_sentences(text)

    for sentence in sentences:
        if len(sentence.split()) >= 10:
            jurafsky_triplets.append({
                'query': sentence,
                'positive': text,
                'negatives': [] # Will add negatives later
            })

print(f"✅ Generated {len(jurafsky_triplets)} Jurafsky triplets")

# 6. Prepare Negative Pool
print("🔄 Preparing negative pool...")

# Load ArXiv data for negatives
arxiv_file = project_root / "data" / "processed" / "arxiv_cs_papers_processed.csv"
arxiv_df = pd.read_csv(arxiv_file)
arxiv_df['combined_text'] = arxiv_df.apply(
    lambda row: f"Title: {row['title']}. Abstract: {row['abstract']}" if pd.notna(row['title']) and pd.notna(row['abstract']) else "",
    axis=1
)

# Pool includes ArXiv passages + Jurafsky chunks
arxiv_pool = arxiv_df['combined_text'].tolist()
jurafsky_pool = [chunk['text'] for chunk in chunks]
negative_pool = arxiv_pool + jurafsky_pool

print(f"✅ Negative pool size: {len(negative_pool)} passages")

# 7. Add Negatives to ALL Triplets
print("🔄 Adding negatives to all triplets...")

def add_negatives(triplets, neg_pool, n_negatives=1):
    for triplet in triplets:
        negs = []
        while len(negs) < n_negatives:
            neg = random.choice(neg_pool)
            if neg != triplet['positive']:
                negs.append(neg)
        triplet['negatives'] = negs
    return triplets

ms_marco_triplets = add_negatives(ms_marco_triplets, negative_pool)
jurafsky_triplets = add_negatives(jurafsky_triplets, negative_pool)
# ArXiv triplets already have negatives from previous step, but let's ensure they are consistent
# If they don't have negatives, add them (though they should already have them)
if arxiv_triplets and 'negatives' not in arxiv_triplets:
    arxiv_triplets = add_negatives(arxiv_triplets, negative_pool)

print(f"✅ Added negatives to all triplets")

# 8. Combine ALL Triplets
all_triplets = arxiv_triplets + ms_marco_triplets + jurafsky_triplets
print(f"\n📊 Total Combined Triplets: {len(all_triplets)}")
print(f"   - ArXiv: {len(arxiv_triplets)}")
print(f"   - MS MARCO: {len(ms_marco_triplets)}")
print(f"   - Jurafsky: {len(jurafsky_triplets)}")

# 9. Split into Train/Val (90/10)
train_triplets, val_triplets = train_test_split(
    all_triplets,
    test_size=0.1,
    random_state=42
)

print(f"\n📂 Final Split:")
print(f"   - Training Triplets: {len(train_triplets)}")
print(f"   - Validation Triplets: {len(val_triplets)}")

# 10. Save for inspection
OUT_COMBINED_TRAIN = project_root / "data" / "processed" / "combined_train_triplets.jsonl"
OUT_COMBINED_VAL   = project_root / "data" / "processed" / "combined_val_triplets.jsonl"

def write_triplets(triplets, path):
    with open(path, 'w', encoding='utf-8') as f:
        for t in triplets:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    print(f"  wrote {len(triplets):,} triplets → {path.name}")

write_triplets(train_triplets, OUT_COMBINED_TRAIN)
write_triplets(val_triplets, OUT_COMBINED_VAL)

print("✅ Data preparation complete! Ready for training.")


📚 Loading MS MARCO data directly from JSON files...
✅ Loaded 20000 queries
✅ Loaded 10000 passages
✅ Loaded 20000 relevance judgments
🔄 Generating MS MARCO triplets...
✅ Generated 20000 MS MARCO pairs
📚 Loading ArXiv triplets...
✅ Loaded 13500 ArXiv triplets
📚 Generating Jurafsky Book triplets...
✅ Generated 11607 Jurafsky triplets
🔄 Preparing negative pool...
✅ Negative pool size: 419061 passages
🔄 Adding negatives to all triplets...
✅ Added negatives to all triplets

📊 Total Combined Triplets: 45107
   - ArXiv: 13500
   - MS MARCO: 20000
   - Jurafsky: 11607

📂 Final Split:
   - Training Triplets: 40596
   - Validation Triplets: 4511
  wrote 40,596 triplets → combined_train_triplets.jsonl
  wrote 4,511 triplets → combined_val_triplets.jsonl
✅ Data preparation complete! Ready for training.


In [ ]:
import pandas as pd
import numpy as np

ARXIV_CANDIDATES = [
    project_root / "data" / "processed" / "arxiv_cs_papers_processed.csv",
    Path("/content/drive/MyDrive/NLP_project/neural_search_real_pipeline_restored/data/processed/arxiv_cs_papers_processed.csv"),
]

ARXIV_FILE = None
for p in ARXIV_CANDIDATES:
    if p.exists():
        ARXIV_FILE = p
        break

print("ArXiv file:", ARXIV_FILE)

if ARXIV_FILE is None:
    raise FileNotFoundError("Could not find arxiv_cs_papers_processed.csv in repo or Drive path.")

# Quick size check only. Full read happens in triplet generation.
sample_arxiv = pd.read_csv(ARXIV_FILE, usecols=["title", "abstract"], nrows=5)
print(sample_arxiv.head())


📉 Reduced from 45107 to 25250 unique triplets
✅ Overlap between Train and Val queries: 0
✅ No leakage detected. Safe to train.


In [ ]:
import json
import random
import re
from pathlib import Path
import pandas as pd

random.seed(42)

# Smaller than previous 200k, so training is faster.
MAX_ARXIV_TRIPLETS = 40000
MAX_MSMARCO_TRIPLETS = 5000
N_NEGATIVES = 4

print("📚 Loading MS MARCO data directly from JSON files...")

QUERIES_FILE = project_root / "data" / "processed" / "msmarco_small_queries.json"
PASSAGES_FILE = project_root / "data" / "processed" / "msmarco_small_passages.json"
QRELS_FILE = project_root / "data" / "processed" / "msmarco_small_qrels.json"

for path in [QUERIES_FILE, PASSAGES_FILE, QRELS_FILE]:
    print(path.name, "exists:", path.exists())

ms_marco_triplets = []
passages_list = []

if QUERIES_FILE.exists() and PASSAGES_FILE.exists() and QRELS_FILE.exists():
    with open(QUERIES_FILE, "r", encoding="utf-8") as f:
        queries_list = json.load(f)

    with open(PASSAGES_FILE, "r", encoding="utf-8") as f:
        passages_list = json.load(f)

    with open(QRELS_FILE, "r", encoding="utf-8") as f:
        qrels = json.load(f)

    query_lookup = {str(q["id"]): q["text"] for q in queries_list}
    passage_lookup = {str(p["id"]): p["text"] for p in passages_list}

    # Detect synthetic MS MARCO-like data. Use it only lightly.
    first_passage_text = passages_list[0].get("text", "").lower() if passages_list else ""
    if "fundamental concept in modern computer science" in first_passage_text:
        print("⚠️ MS MARCO file looks synthetic. Using it lightly only.")

    for query_id, relevant_passages in qrels.items():
        query_id = str(query_id)

        if query_id not in query_lookup or not relevant_passages:
            continue

        query_text = query_lookup[query_id].strip()

        best_pid, best_score = max(relevant_passages, key=lambda x: x[1])
        best_pid = str(best_pid)

        if best_pid not in passage_lookup:
            continue

        positive_text = passage_lookup[best_pid].strip()

        if len(query_text.split()) < 3 or len(positive_text.split()) < 10:
            continue

        ms_marco_triplets.append({
            "query": query_text,
            "positive": positive_text,
            "negatives": [],
            "source": "msmarco",
        })

    random.shuffle(ms_marco_triplets)
    ms_marco_triplets = ms_marco_triplets[:MAX_MSMARCO_TRIPLETS]

print(f"✅ MS MARCO triplets used: {len(ms_marco_triplets)}")

print("📚 Loading ArXiv CSV and creating ArXiv title→abstract triplets...")

arxiv_df = pd.read_csv(ARXIV_FILE, usecols=["title", "abstract"])
arxiv_df = arxiv_df.dropna(subset=["title", "abstract"])

arxiv_df["title"] = arxiv_df["title"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
arxiv_df["abstract"] = arxiv_df["abstract"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

arxiv_df = arxiv_df[
    (arxiv_df["title"].str.len() > 10) &
    (arxiv_df["abstract"].str.len() > 120)
]

if len(arxiv_df) > MAX_ARXIV_TRIPLETS:
    arxiv_df = arxiv_df.sample(n=MAX_ARXIV_TRIPLETS, random_state=42)

arxiv_df["combined_text"] = (
    "Title: " + arxiv_df["title"] +
    ". Abstract: " + arxiv_df["abstract"]
)

arxiv_texts = arxiv_df["combined_text"].tolist()
arxiv_titles = arxiv_df["title"].tolist()

arxiv_triplets = [
    {
        "query": title,
        "positive": full_text,
        "negatives": [],
        "source": "arxiv",
    }
    for title, full_text in zip(arxiv_titles, arxiv_texts)
]

print(f"✅ ArXiv triplets used: {len(arxiv_triplets)}")

print("📚 Generating Jurafsky sentence→chunk triplets...")

def extract_sentences(text):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [
        s.strip()
        for s in sentences
        if len(s.split()) >= 8
        and len(s.split()) <= 40
        and not s.strip().startswith("(")
        and not s.strip().startswith("[")
    ]

jurafsky_triplets = []

for chunk in chunks:
    text = chunk.get("text", "")

    if not isinstance(text, str) or not text.strip():
        continue

    sentences = extract_sentences(text)

    # Use up to 3 useful sentence queries per chunk, not every sentence.
    # This keeps data smaller and more diverse.
    if len(sentences) > 3:
        sentences = random.sample(sentences, 3)

    # Also add a generic chunk-start query, useful for textbook search.
    first_words = " ".join(text.split()[:18])
    candidate_queries = sentences + [first_words]

    for q in candidate_queries:
        jurafsky_triplets.append({
            "query": q,
            "positive": text,
            "negatives": [],
            "source": "jurafsky",
        })

print(f"✅ Jurafsky triplets generated: {len(jurafsky_triplets)}")

print("🔄 Preparing negative pools...")

jurafsky_pool = [
    c["text"]
    for c in chunks
    if isinstance(c.get("text", ""), str) and len(c["text"].split()) > 80
]

msmarco_pool = [
    p["text"]
    for p in passages_list
    if isinstance(p.get("text", ""), str) and len(p["text"].split()) > 20
]

arxiv_pool = arxiv_texts

all_negative_pool = list(dict.fromkeys(arxiv_pool + jurafsky_pool + msmarco_pool))

print(f"Negative pools:")
print(f"   - ArXiv:    {len(arxiv_pool)}")
print(f"   - Jurafsky: {len(jurafsky_pool)}")
print(f"   - MS MARCO: {len(msmarco_pool)}")
print(f"   - All:      {len(all_negative_pool)}")

def normalize_negatives(negs):
    if isinstance(negs, str):
        return [negs]
    if isinstance(negs, list):
        return [n for n in negs if isinstance(n, str) and n.strip()]
    return []

def add_negatives(triplets, same_domain_pool, all_pool, n_negatives=4):
    fixed = []

    for triplet in triplets:
        query = triplet.get("query", "")
        positive = triplet.get("positive", "")

        if not isinstance(query, str) or not isinstance(positive, str):
            continue

        query = query.strip()
        positive = positive.strip()

        if not query or not positive:
            continue

        existing = normalize_negatives(triplet.get("negatives", []))

        # Add at least 1 same-domain negative when possible.
        tries = 0
        while len(existing) < 1 and same_domain_pool and tries < 100:
            neg = random.choice(same_domain_pool)
            tries += 1
            if neg != positive and neg not in existing:
                existing.append(neg)

        # Fill the rest with mixed-domain negatives.
        tries = 0
        while len(existing) < n_negatives and tries < 300:
            neg = random.choice(all_pool)
            tries += 1
            if neg != positive and neg not in existing:
                existing.append(neg)

        if len(existing) >= n_negatives:
            new_t = dict(triplet)
            new_t["query"] = query
            new_t["positive"] = positive
            new_t["negatives"] = existing[:n_negatives]
            fixed.append(new_t)

    return fixed

print("🔄 Adding negatives...")

arxiv_triplets = add_negatives(arxiv_triplets, arxiv_pool, all_negative_pool, N_NEGATIVES)
ms_marco_triplets = add_negatives(ms_marco_triplets, msmarco_pool, all_negative_pool, N_NEGATIVES)
jurafsky_triplets = add_negatives(jurafsky_triplets, jurafsky_pool, all_negative_pool, N_NEGATIVES)

all_triplets_raw = arxiv_triplets + ms_marco_triplets + jurafsky_triplets

print("\n📊 Triplets before split:")
print(f"   - ArXiv:     {len(arxiv_triplets)}")
print(f"   - MS MARCO:  {len(ms_marco_triplets)}")
print(f"   - Jurafsky:  {len(jurafsky_triplets)}")
print(f"   - TOTAL:     {len(all_triplets_raw)}")

bad = [
    t for t in all_triplets_raw
    if "negatives" not in t
    or not isinstance(t["negatives"], list)
    or len(t["negatives"]) < N_NEGATIVES
]

print(f"Bad triplets without {N_NEGATIVES} negatives: {len(bad)}")
if bad:
    raise ValueError("Some triplets still have missing/bad negatives.")

OUT_ALL_TRIPLETS = project_root / "data" / "processed" / "combined_all_triplets.jsonl"
with open(OUT_ALL_TRIPLETS, "w", encoding="utf-8") as f:
    for t in all_triplets_raw:
        f.write(json.dumps(t, ensure_ascii=False) + "\n")

print(f"✅ Saved all raw triplets to: {OUT_ALL_TRIPLETS}")


🧠 Initializing Encoder Model...
🚀 Using device: cpu
🔄 Fitting Tokenizer on training data...
✅ Vocabulary size: 20000
🚀 Starting Training...
Epoch 1/2 | Batch 0 | Loss: 0.7085
Epoch 1/2 | Batch 50 | Loss: 0.6532
🏁 Epoch 1 Complete | Avg Loss: 0.6606
📊 Validation Loss: 0.7173
Epoch 2/2 | Batch 0 | Loss: 0.6004
Epoch 2/2 | Batch 50 | Loss: 0.5309
🏁 Epoch 2 Complete | Avg Loss: 0.5433
📊 Validation Loss: 0.1591

✅ Training Complete in 5.0 minutes!
💾 Saving model to c:\Users\salo\NLP_final_project\models\encoder_transformer.pth...
Model saved to c:\Users\salo\NLP_final_project\models\encoder_transformer.pth
✅ Model Saved!


In [ ]:
import random
import json
from collections import Counter
from sklearn.model_selection import train_test_split

print("🔄 Deduplicating, splitting and balancing triplets...")

N_NEGATIVES = 4
JURAFSKY_TRAIN_MULTIPLIER = 5

def clean_one(t):
    query = t.get("query", "")
    positive = t.get("positive", "")
    negatives = t.get("negatives", [])
    source = t.get("source", "unknown")

    if not isinstance(query, str) or not isinstance(positive, str):
        return None
    if not isinstance(negatives, list):
        return None

    query = query.strip()
    positive = positive.strip()

    negatives = [
        n.strip()
        for n in negatives
        if isinstance(n, str) and n.strip() and n.strip() != positive
    ]

    if not query or not positive or len(negatives) < N_NEGATIVES:
        return None

    return {
        "query": query,
        "positive": positive,
        "negatives": negatives[:N_NEGATIVES],
        "source": source,
    }

clean_triplets = []
seen = set()

for t in all_triplets_raw:
    ct = clean_one(t)
    if ct is None:
        continue

    key = (ct["query"], ct["positive"], ct["source"])
    if key in seen:
        continue

    seen.add(key)
    clean_triplets.append(ct)

print(f"Clean unique triplets: {len(clean_triplets)}")
print("Source counts:", Counter(t["source"] for t in clean_triplets))

by_source = {}
for t in clean_triplets:
    by_source.setdefault(t["source"], []).append(t)

train_triplets_base = []
val_triplets = []

for source, items in by_source.items():
    random.seed(42)
    random.shuffle(items)

    if len(items) < 10:
        train_items = items
        val_items = []
    else:
        train_items, val_items = train_test_split(
            items,
            test_size=0.05,
            random_state=42
        )

    train_triplets_base.extend(train_items)
    val_triplets.extend(val_items)

# Oversample Jurafsky only in training, after split.
jur_train = [t for t in train_triplets_base if t["source"] == "jurafsky"]
non_jur_train = [t for t in train_triplets_base if t["source"] != "jurafsky"]

train_triplets = non_jur_train + (jur_train * JURAFSKY_TRAIN_MULTIPLIER)

random.seed(42)
random.shuffle(train_triplets)
random.shuffle(val_triplets)

print("\n📂 Final Split:")
print(f"   - Base train before oversampling: {len(train_triplets_base)}")
print(f"   - Training Triplets:             {len(train_triplets)}")
print(f"   - Validation Triplets:           {len(val_triplets)}")
print("Train source counts:", Counter(t["source"] for t in train_triplets))
print("Val source counts:", Counter(t["source"] for t in val_triplets))

OUT_COMBINED_TRAIN = project_root / "data" / "processed" / "combined_train_triplets.jsonl"
OUT_COMBINED_VAL = project_root / "data" / "processed" / "combined_val_triplets.jsonl"

def write_triplets(triplets, path):
    with open(path, "w", encoding="utf-8") as f:
        for t in triplets:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    print(f"  wrote {len(triplets):,} triplets → {path}")

write_triplets(train_triplets, OUT_COMBINED_TRAIN)
write_triplets(val_triplets, OUT_COMBINED_VAL)

print("✅ Data split complete.")


🧠 Loading Model...
Model loaded from C:\Users\salo\NLP_final_project\models\encoder_transformer.pth
🔄 Fitting Tokenizer...
📚 Loading & Encoding Passages...
yellow
aaa
✅ Encoded 1730 passages

🔍 INTERACTIVE NEURAL SEARCH
Type 'quit' to exit.

ssssss
[[1623 1559  168 ...  225 1699  520]]
tion, Duisburg, Germany, 24–26 March, pp. 72–81. Mitamura, T. and Nyberg, E. H. (1995). Controlled English fo r knowledge-based MT: Experience with the KANT system. In 6th International Conference on The- oretical and Methodological Issues in Machine Translation. Mitchell, D. C., Cuetos, F., Corley, M. M. B., and Brysbaert,M. (1995). Exposure- based models of human parsing: Evidence for the use of coarse -grained (nonlexi- cal) statistical records. Journal of Psycholinguistic Research, 24(6), 469–488. Mitchell, T. M. (1981). Generalization as search. In Webber , B. L. and Nilsson, N. J. (Eds.), Readings in Artiﬁcial Intelligence, pp. 517–542. Morgan Kaufmann, Los Altos. Mitkov, R. and Boguraev, B. (Eds.)

In [ ]:
import torch
import torch.optim as optim
import random
import numpy as np
from pathlib import Path
import sys
import time
import json
import importlib
import shutil

# Always reload changed files from disk.
import src.models.encoder as encoder_module
import src.models.loss as loss_module

importlib.reload(encoder_module)
importlib.reload(loss_module)

EncoderOnlyTransformer = encoder_module.EncoderOnlyTransformer
SimpleTokenizer = encoder_module.SimpleTokenizer
InfoNCELoss = loss_module.InfoNCELoss

CONFIG = {
    "vocab_size": 50000,
    "d_model": 128,
    "n_heads": 4,
    "n_layers": 3,
    "d_ff": 512,
    "max_len": 192,
    "dropout": 0.2,
    "pad_token_id": 0
}

TRAINING_CONFIG = {
    "epochs": 4,
    "batch_size": 128,
    "learning_rate": 1e-4,
    "temperature": 0.07,
    "weight_decay": 1e-4,
    "n_negatives": 4
}

MODEL_SAVE_PATH = project_root / "models" / "encoder_transformer.pth"
TOKENIZER_SAVE_PATH = project_root / "models" / "tokenizer_vocab.json"

MODEL_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Training triplets:", len(train_triplets))
print("Validation triplets:", len(val_triplets))

if len(train_triplets) == 0:
    raise ValueError("train_triplets is empty. Run triplet and split cells first.")

N_NEGATIVES = TRAINING_CONFIG["n_negatives"]

bad_train = [
    t for t in train_triplets
    if "negatives" not in t
    or not isinstance(t["negatives"], list)
    or len(t["negatives"]) < N_NEGATIVES
]

print("Bad train triplets:", len(bad_train))

if bad_train:
    raise ValueError("Some training triplets have bad negatives.")

print("🧠 Initializing Encoder Model...")
model = EncoderOnlyTransformer(CONFIG)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to_device(device)

print(f"🚀 Using device: {device}")

print("🔄 Fitting tokenizer on train data...")

all_texts_for_vocab = []

for t in train_triplets:
    all_texts_for_vocab.append(t["query"])
    all_texts_for_vocab.append(t["positive"])

    for neg in t["negatives"]:
        if isinstance(neg, str) and neg.strip():
            all_texts_for_vocab.append(neg)

tokenizer = SimpleTokenizer(CONFIG["vocab_size"])
tokenizer.fit(all_texts_for_vocab)

print(f"✅ Vocabulary size: {len(tokenizer.word_to_id)}")

tokenizer.save(str(TOKENIZER_SAVE_PATH))
print(f"✅ Saved tokenizer to: {TOKENIZER_SAVE_PATH}")

def get_negatives(t, n_negatives):
    negs = t.get("negatives", [])

    if isinstance(negs, str):
        negs = [negs]

    negs = [
        n for n in negs
        if isinstance(n, str) and n.strip()
    ]

    if len(negs) < n_negatives:
        raise ValueError("Triplet does not have enough negatives.")

    return negs[:n_negatives]

def generate_batches(triplets, tokenizer, batch_size, max_len, n_negatives, shuffle=True):
    triplets = list(triplets)

    if shuffle:
        random.shuffle(triplets)

    for i in range(0, len(triplets), batch_size):
        batch_triplets = triplets[i:i + batch_size]

        queries = [t["query"] for t in batch_triplets]
        positives = [t["positive"] for t in batch_triplets]

        negatives_nested = [
            get_negatives(t, n_negatives)
            for t in batch_triplets
        ]

        flat_negatives = [
            neg
            for negs in negatives_nested
            for neg in negs
        ]

        q_encoded = tokenizer(
            queries,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )

        p_encoded = tokenizer(
            positives,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )

        n_encoded = tokenizer(
            flat_negatives,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )

        yield {
            "query_ids": q_encoded["input_ids"],
            "query_mask": q_encoded["attention_mask"],
            "pos_ids": p_encoded["input_ids"],
            "pos_mask": p_encoded["attention_mask"],
            "neg_ids": n_encoded["input_ids"],
            "neg_mask": n_encoded["attention_mask"],
            "batch_size_actual": len(batch_triplets)
        }

print("🚀 Starting Training...")
print("=" * 50)

optimizer = optim.AdamW(
    model.parameters(),
    lr=TRAINING_CONFIG["learning_rate"],
    weight_decay=TRAINING_CONFIG["weight_decay"]
)

criterion = InfoNCELoss(temperature=TRAINING_CONFIG["temperature"])

num_epochs = TRAINING_CONFIG["epochs"]
batch_size = TRAINING_CONFIG["batch_size"]
max_len = CONFIG["max_len"]
n_negatives = TRAINING_CONFIG["n_negatives"]

start_time = time.time()

for epoch in range(num_epochs):
    model.train()

    epoch_loss = 0.0
    num_batches = 0

    data_gen = generate_batches(
        train_triplets,
        tokenizer,
        batch_size,
        max_len,
        n_negatives,
        shuffle=True
    )

    for batch_idx, batch in enumerate(data_gen):
        q_ids = batch["query_ids"].to(device)
        q_mask = batch["query_mask"].to(device)

        p_ids = batch["pos_ids"].to(device)
        p_mask = batch["pos_mask"].to(device)

        n_ids = batch["neg_ids"].to(device)
        n_mask = batch["neg_mask"].to(device)

        bsz = batch["batch_size_actual"]

        anchor_emb = model(q_ids, q_mask)
        pos_emb = model(p_ids, p_mask)

        neg_emb_flat = model(n_ids, n_mask)
        neg_emb = neg_emb_flat.view(bsz, n_negatives, CONFIG["d_model"])

        loss = criterion(anchor_emb, pos_emb, neg_emb)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

        if batch_idx % 100 == 0:
            print(
                f"Epoch {epoch + 1}/{num_epochs} | "
                f"Batch {batch_idx} | "
                f"Loss: {loss.item():.4f}"
            )

    avg_loss = epoch_loss / max(num_batches, 1)
    print(f"🏁 Epoch {epoch + 1} Complete | Avg Train Loss: {avg_loss:.4f}")

    model.eval()
    val_loss = 0.0
    val_batches = 0

    val_gen = generate_batches(
        val_triplets,
        tokenizer,
        batch_size,
        max_len,
        n_negatives,
        shuffle=False
    )

    with torch.no_grad():
        for batch in val_gen:
            v_q = batch["query_ids"].to(device)
            v_q_mask = batch["query_mask"].to(device)

            v_p = batch["pos_ids"].to(device)
            v_p_mask = batch["pos_mask"].to(device)

            v_n = batch["neg_ids"].to(device)
            v_n_mask = batch["neg_mask"].to(device)

            bsz = batch["batch_size_actual"]

            v_anchor = model(v_q, v_q_mask)
            v_pos = model(v_p, v_p_mask)

            v_neg_flat = model(v_n, v_n_mask)
            v_neg = v_neg_flat.view(bsz, n_negatives, CONFIG["d_model"])

            v_loss = criterion(v_anchor, v_pos, v_neg)

            val_loss += v_loss.item()
            val_batches += 1

    avg_val_loss = val_loss / max(val_batches, 1)
    print(f"📊 Validation Loss: {avg_val_loss:.4f}")

total_time = time.time() - start_time

print(f"\n✅ Training complete in {total_time / 60:.1f} minutes")

print(f"💾 Saving model to {MODEL_SAVE_PATH}")
model.save(str(MODEL_SAVE_PATH))
print("✅ Model saved")

print(f"✅ Tokenizer saved to {TOKENIZER_SAVE_PATH}")

# Save to Drive so session loss does not delete the trained model.
DRIVE_MODEL_DIR = Path("/content/drive/MyDrive/NLP_project/saved_models")
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(MODEL_SAVE_PATH, DRIVE_MODEL_DIR / "encoder_transformer.pth")
shutil.copy2(TOKENIZER_SAVE_PATH, DRIVE_MODEL_DIR / "tokenizer_vocab.json")

print(f"✅ Copied model/tokenizer to Drive: {DRIVE_MODEL_DIR}")


In [ ]:
import random
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity

model.eval()

sample = random.sample(val_triplets, min(500, len(val_triplets)))

correct_at_1 = 0
mrr_total = 0.0
jur_correct_at_1 = 0
jur_total = 0

for t in sample:
    query = t["query"]
    candidates = [t["positive"]] + t["negatives"][:4]

    with torch.no_grad():
        q_emb = model.encode([query], tokenizer)
        c_emb = model.encode(candidates, tokenizer)

    scores = cosine_similarity(q_emb, c_emb)[0]
    ranked = np.argsort(scores)[::-1]

    rank_of_positive = list(ranked).index(0) + 1

    if rank_of_positive == 1:
        correct_at_1 += 1
        if t.get("source") == "jurafsky":
            jur_correct_at_1 += 1

    if t.get("source") == "jurafsky":
        jur_total += 1

    mrr_total += 1.0 / rank_of_positive

print("Random Positive@1 baseline with 5 candidates: 0.20")
print("Positive@1 accuracy:", correct_at_1 / len(sample))
print("MRR:", mrr_total / len(sample))

if jur_total > 0:
    print("Jurafsky Positive@1 accuracy in sample:", jur_correct_at_1 / jur_total)
else:
    print("No Jurafsky examples in sampled validation subset.")


In [ ]:
import json
import sys
import numpy as np
import torch
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
import importlib

PROJECT_ROOT = Path("/content/NLP_final_project").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.models.encoder as encoder_module
importlib.reload(encoder_module)

EncoderOnlyTransformer = encoder_module.EncoderOnlyTransformer
SimpleTokenizer = encoder_module.SimpleTokenizer

MODEL_PATH = PROJECT_ROOT / "models" / "encoder_transformer.pth"
TOKENIZER_PATH = PROJECT_ROOT / "models" / "tokenizer_vocab.json"
CHUNKS_PATH = PROJECT_ROOT / "data" / "jurafsky_chunks" / "chunks.jsonl"

# Fallback to Drive if runtime model is missing.
DRIVE_MODEL_DIR = Path("/content/drive/MyDrive/NLP_project/saved_models")

if not MODEL_PATH.exists() and (DRIVE_MODEL_DIR / "encoder_transformer.pth").exists():
    MODEL_PATH = DRIVE_MODEL_DIR / "encoder_transformer.pth"

if not TOKENIZER_PATH.exists() and (DRIVE_MODEL_DIR / "tokenizer_vocab.json").exists():
    TOKENIZER_PATH = DRIVE_MODEL_DIR / "tokenizer_vocab.json"

print("Model path:", MODEL_PATH, "exists:", MODEL_PATH.exists())
print("Tokenizer path:", TOKENIZER_PATH, "exists:", TOKENIZER_PATH.exists())
print("Chunks path:", CHUNKS_PATH, "exists:", CHUNKS_PATH.exists())

if not MODEL_PATH.exists():
    raise FileNotFoundError("Model not found. Run training cell first.")
if not TOKENIZER_PATH.exists():
    raise FileNotFoundError("Tokenizer not found. Run training cell first.")
if not CHUNKS_PATH.exists():
    raise FileNotFoundError("Jurafsky chunks not found. Run chunk rebuild cell first.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("🧠 Loading model...")
model = EncoderOnlyTransformer.load(str(MODEL_PATH), device=device)
model.eval()
print("✅ Model loaded on:", device)

print("🔄 Loading tokenizer...")
tokenizer = SimpleTokenizer.load(str(TOKENIZER_PATH))
print("✅ Tokenizer loaded")
print("Vocab size:", len(tokenizer.word_to_id))

def is_good_search_chunk(text):
    if not isinstance(text, str):
        return False

    text = text.strip()
    words = text.split()

    if len(words) < 80:
        return False

    alpha_chars = sum(ch.isalpha() for ch in text)
    if alpha_chars < 250:
        return False

    lower = text.lower()

    bad_markers = [
        "acknowledg",
        "bibliography",
        "references",
        "author index",
        "subject index",
        "index",
        "figure c.",
        "tag description example",
        "ucrel",
    ]

    if any(marker in lower[:500] for marker in bad_markers):
        return False

    if text.count(",") > 70 and len(words) < 260:
        return False

    return True

print("📚 Loading Jurafsky passages only...")

passages = []

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        try:
            data = json.loads(line)
        except json.JSONDecodeError:
            continue

        text = data.get("text", "")

        if is_good_search_chunk(text):
            passages.append(text.strip())

print(f"✅ Loaded {len(passages)} clean Jurafsky passages")

if not passages:
    raise ValueError("No good Jurafsky passages loaded. Check chunking/filtering.")

print("🔄 Encoding Jurafsky passages...")

batch_size = 128
embeddings = []

for i in range(0, len(passages), batch_size):
    batch = passages[i:i + batch_size]

    with torch.no_grad():
        emb = model.encode(batch, tokenizer)

    embeddings.append(emb)

    if i % (batch_size * 20) == 0:
        print(f"Encoded {i}/{len(passages)}")

passage_embeddings = np.vstack(embeddings)

print(f"✅ Encoded {len(passages)} passages")
print("Embedding shape:", passage_embeddings.shape)

print("\n" + "=" * 50)
print("🔍 JURAFSKY-ONLY NEURAL SEARCH")
print("=" * 50)
print("Type 'quit' to exit.\n")

while True:
    try:
        query = input("Enter query: ")
    except EOFError:
        break

    query = query.strip()

    if query.lower() == "quit":
        print("👋 Exiting...")
        break

    if not query:
        continue

    with torch.no_grad():
        query_emb = model.encode([query], tokenizer)

    similarities = cosine_similarity(query_emb, passage_embeddings)[0]

    top_indices = np.argsort(similarities)[::-1][:5]

    print("\nTop 5 results:")

    for rank, idx in enumerate(top_indices, start=1):
        print("\n" + "-" * 90)
        print(f"Rank {rank} | score={similarities[idx]:.4f}")
        print(passages[idx][:1400])

print("✅ Search complete")
